In [1]:

#  Imports
from pyspark.sql import SparkSession
from pyspark.sql.types import *
import socket, json, threading, os
from datetime import datetime, timezone
from queue import Queue, Empty

RAW_OUTPUT_DIR = "./data/raw"
os.makedirs(RAW_OUTPUT_DIR, exist_ok=True)

PRODUCER_HOST = "localhost"   # change if producer runs on another machine
PRODUCER_PORT = 9999

In [2]:
# Spark Session
spark = SparkSession.builder \
    .appName("WorldCupFirehose") \
    .config("spark.sql.shuffle.partitions", "4") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")
print("[Spark] Session ready.")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/02 12:39:36 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


[Spark] Session ready.


In [3]:
# Schema definition (mirrors producer's output)
RAW_SCHEMA = StructType([
    StructField("did",         StringType(),  True),
    StructField("post_uri",    StringType(),  True),
    StructField("text",        StringType(),  False),
    StructField("created_at",  StringType(),  True),   # ISO 8601 string
    StructField("ingested_at", StringType(),  True),
    StructField("reply_to",    StringType(),  True),   # nullable
    StructField("mentions",    ArrayType(StringType()), True),
    StructField("urls",        ArrayType(StringType()), True),
    StructField("hashtags",    ArrayType(StringType()), True),
    StructField("lang",        StringType(),  True),
])

In [4]:
# TCP receiver thread
# Reads newline-delimited JSON from the producer and puts parsed dicts
# into a thread-safe queue for Spark to consume in batches.

post_queue = Queue()

def tcp_receiver():
    while True:
        try:
            print(f"[Consumer] Connecting to producer at {PRODUCER_HOST}:{PRODUCER_PORT}...")
            with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
                s.connect((PRODUCER_HOST, PRODUCER_PORT))
                print("[Consumer] Connected to producer.")
                buffer = ""
                while True:
                    chunk = s.recv(4096).decode("utf-8")
                    if not chunk:
                        break
                    buffer += chunk
                    while "\n" in buffer:
                        line, buffer = buffer.split("\n", 1)
                        line = line.strip()
                        if line:
                            try:
                                post_queue.put(json.loads(line))
                            except json.JSONDecodeError:
                                pass
        except Exception as e:
            print(f"[Consumer] Connection error: {e}. Retrying in 5s...")
            import time; time.sleep(5)

receiver_thread = threading.Thread(target=tcp_receiver, daemon=True)
receiver_thread.start()
print("[Consumer] Receiver thread started.")

[Consumer] Connecting to producer at localhost:9999...
[Consumer] Receiver thread started.
[Consumer] Connected to producer.


In [5]:
# Micro-batch ingestion loop
# Every BATCH_INTERVAL seconds, drains the queue, creates a Spark DataFrame,
# registers/refreshes the "raw" temp view, and appends to disk as Parquet.

import time

BATCH_INTERVAL = 5      # seconds between micro-batches
batch_number   = 0
all_rows       = []     # accumulates all rows for the global "raw" table

print("[Consumer] Starting ingestion loop. Stop the cell to halt.")

while True:
    time.sleep(BATCH_INTERVAL)

    # Drain everything currently in the queue
    batch_rows = []
    while True:
        try:
            batch_rows.append(post_queue.get_nowait())
        except Empty:
            break

    if not batch_rows:
        print(f"[Batch {batch_number}] No new posts yet...")
        continue

    batch_number += 1
    all_rows.extend(batch_rows)

    # Build Spark DataFrame for this batch
    batch_df = spark.createDataFrame(batch_rows, schema=RAW_SCHEMA)

    # Register the FULL corpus as the "raw" temp view (Person B reads this)
    full_df = spark.createDataFrame(all_rows, schema=RAW_SCHEMA)
    full_df.createOrReplaceTempView("raw")

    # Append this batch to disk (Parquet, partitioned by date)
    batch_df.write \
        .mode("append") \
        .parquet(RAW_OUTPUT_DIR)

    print(f"[Batch {batch_number}] +{len(batch_rows)} posts | {len(all_rows)} total in raw | saved to {RAW_OUTPUT_DIR}")

[Consumer] Starting ingestion loop. Stop the cell to halt.


26/06/02 12:39:55 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/06/02 12:39:55 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 84.44% for 9 writers
26/06/02 12:39:55 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers


[Batch 1] +8 posts | 8 total in raw | saved to ./data/raw


[Batch 2] +25 posts | 33 total in raw | saved to ./data/raw


[Batch 3] +17 posts | 50 total in raw | saved to ./data/raw


[Batch 4] +22 posts | 72 total in raw | saved to ./data/raw


[Batch 5] +10 posts | 82 total in raw | saved to ./data/raw


[Batch 6] +8 posts | 90 total in raw | saved to ./data/raw
[Batch 7] +14 posts | 104 total in raw | saved to ./data/raw


[Batch 8] +15 posts | 119 total in raw | saved to ./data/raw
[Batch 9] +16 posts | 135 total in raw | saved to ./data/raw


KeyboardInterrupt: 